# Poster campaign · 1/3 — Setup

Prepare one **SI-only O1, enriched-Ge, full-background** physics calculation, using the optimizations on `poster-speedup` after merge into the notebook branch. This creates a **dedicated campaign configuration**; the original `analyses/legend_poster/config.json` is not overwritten.

The three background selections (`none`, `neutrinos`, `full`), three analysis thresholds (**100, 150, 1000 eVee**) and three exposures (**0.1, 1, 10 tonne-years**) are applied **after detector folding**. They give 27 statistical combinations but only one expensive source-spectrum calculation.

**Assumptions:** thresholds are analysis cuts on a *single fixed* response/efficiency model. Exposure merely scales rates; time evolution of cosmogenic activities is not recomputed. The `legend_poster` source model and enriched composition are provisional. This notebook does not certify either optimization scientifically.

In [1]:
from pathlib import Path
import sys

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'analyses/legend_poster').is_dir():
            return candidate
    raise RuntimeError('Open this notebook in the WIMP-LEGEND checkout (or a subdirectory).')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# CAMPAIGN_DIR = ROOT / 'analyses/legend_poster/campaigns/poster_final'
CAMPAIGN_DIR = ROOT / 'analyses/legend_poster/campaigns/poster_sigma20'
SPEC_PATH = ROOT / 'analyses/legend_poster/poster_campaign_spec.json'
CONFIG_PATH = CAMPAIGN_DIR / 'config.json'
print('Repository:', ROOT)
print('Campaign:', CAMPAIGN_DIR)


Repository: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code
Campaign: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code\analyses\legend_poster\campaigns\poster_sigma20


## Input assumptions and dense mass grid

This inherits all physics and numerical tolerances from the repository's saved master configuration. We change only the campaign dimensions and select the optional validated rate/folding backends. Keep an explicit 2/5/10 GeV subset for the spectra plot.

In [2]:
import json
from copy import deepcopy
import numpy as np
from analyses.legend_poster import load_config, validate_config, resolve_profile
from analyses.legend_poster.analysis import observed_edges
from analyses.legend_poster.campaign_scripts.poster_campaign import load_spec

spec = load_spec(SPEC_PATH)
base = load_config(
    ROOT / 'analyses/legend_poster/campaigns/poster_final/config.json'
)
working = deepcopy(base)

# Test: change baseline noise from 50 to 20 eVee
working['detector']['presets']['benchmark']['noise_sigma_keV'] = 0.02 

# Edit these choices to reduce the pilot scan, WITHOUT changing physics precision.
MASS_MODE = 'final'  # 'final' or 'pilot' (3 masses, same numerical tolerances)
final_masses = np.unique(np.r_[np.geomspace(0.5, 10.0, 48),
                               np.geomspace(10.0, 100.0, 30),
                               spec['spectrum_masses_GeV']])
working['signal']['masses_GeV'] = ([2.0, 5.0, 10.0] if MASS_MODE == 'pilot'
                                  else final_masses.tolist())
working['signal']['operators'] = [1]
working['signal']['si_rate_engine'] = 'factorized_shm'
working['numerical']['folding_backend'] = 'vectorized'
working['analysis']['scenarios'] = [spec['input_scenario']]
working['analysis']['methods'] = [spec['statistics_method']]
working['analysis']['exposures_kg_day'] = [0.1 * 1000 * working['analysis']['days_per_year']]
working['background']['historical']['enabled'] = False
working['detector']['presets'][working['detector']['default']]['threshold_keV_obs'] = 0.1
# working["detector"]["presets"]["benchmark"]["noise_sigma_keV"] = 0.05
working['execution']['default_profile'] = 'standard'
working['execution']['profiles']['standard'] = {}  # prevent a profile overriding campaign settings

working = validate_config(working)
resolved = resolve_profile(working, 'standard')
assert resolved['signal']['operators'] == [1]
assert resolved['signal']['si_rate_engine'] == 'factorized_shm'
assert resolved['numerical']['folding_backend'] == 'vectorized'
assert resolved['analysis']['scenarios'] == ['legend_poster']
assert all(np.any(np.isclose(observed_edges(resolved), t, atol=1e-10, rtol=0))
           for t in spec['thresholds_keVee']), 'Thresholds must match observed bin edges'

print('WIMP masses:', len(working['signal']['masses_GeV']),
      'from', min(working['signal']['masses_GeV']), 'to', max(working['signal']['masses_GeV']), 'GeV')
print('Rate:', working['signal']['si_rate_engine'],
      '| folding:', working['numerical']['folding_backend'])
print('Statistical combinations per mass:', len(spec['background_modes']) *
      len(spec['thresholds_keVee']) * len(spec['exposures_tonne_year']))
print('Target:', working['background']['scenarios']['legend_poster']['target'])


WIMP masses: 79 from 0.5 to 100.0 GeV
Rate: factorized_shm | folding: vectorized
Statistical combinations per mass: 27
Target: enriched_provisional


## Inspect assumptions before saving

In particular, confirm the adopted Ge isotope mixture, low-energy efficiency, quenching/noise parameters, source activities and halo model. The additional scan dimensions **must not be confused with physical detector thresholds or time-dependent exposure predictions**.

In [3]:
for name, value in [
    ('Ge target', working['target']['presets']['enriched_provisional']),
    ('Detector', working['detector']['presets'][working['detector']['default']]),
    ('Background', working['background']['base']),
    ('CL and ROI', working['analysis']),
    ('Numerical tolerances', working['numerical'])]:
    print('\n' + name + ':')
    print(json.dumps(value, indent=2)[:4200])



Ge target:
{
  "fraction_kind": "number",
  "fractions": {
    "70": 0.0,
    "72": 0.0,
    "73": 0.0,
    "74": 0.08,
    "76": 0.92
  },
  "provenance": "HYPOTHETICAL 92% Ge76 / 8% Ge74 by number; NOT measured LEGEND enrichment. Replace all fractions with collaboration data.",
  "status": "hypothetical"
}

Detector:
{
  "provenance": "Hypothetical low-energy noise/Fano, constant efficiency and Lindhard assumptions; uncalibrated",
  "threshold_keV_obs": 0.1,
  "noise_sigma_keV": 0.02,
  "fano_factor": 0.11,
  "pair_energy_keV": 0.00296,
  "efficiency": 0.9,
  "lindhard_k": 0.157,
  "Z": 32,
  "NR_energy_range_keV": [
    0.0,
    1500.0
  ],
  "ER_energy_range_keV": [
    0.0,
    20000.0
  ]
}

Background:
{
  "lar_transfer_probability": 0.00014230663956447948,
  "argon_mass_per_kg_Ge": 20.0,
  "cooldown_day": 365.25,
  "run_duration_day": 365.25,
  "ga68_initialization": "secular_equilibrium",
  "ga68_initial_activity_per_kg_day": null,
  "zn65_gamma_escape_probability": 0.0,
  "g

## Explicitly save the dedicated configuration

This does not touch the source master JSON. After the merge, verify you are on the intended code revision before launching the expensive run.

In [4]:
from analyses.legend_poster import save_config, load_config
SAVE_CAMPAIGN_CONFIG = True
if SAVE_CAMPAIGN_CONFIG:
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_config(working, CONFIG_PATH)
    print('Saved:', CONFIG_PATH)
else:
    print('Configuration not saved. Set SAVE_CAMPAIGN_CONFIG=True to save.')


Saved: C:\Users\Tiago\Desktop\Tiago\Studium\HiWi\MPIK\LEGEND\DM Spektra\WIMP Code\analyses\legend_poster\campaigns\poster_sigma20\config.json


In [5]:
config = load_config(CONFIG_PATH)
assert config["signal"]["si_rate_engine"] == "factorized_shm"
assert config["numerical"]["folding_backend"] == "vectorized"